# 🎬 IMDB Sentiment Analysis - Data Preparation & EDA

**Project Overview:**
Building a comprehensive sentiment analysis system for IMDB movie reviews using both traditional ML and modern deep learning approaches.

**Dataset:** IMDB Dataset with 50,000 movie reviews
**Target:** Binary sentiment classification (Positive/Negative)

---

In [ ]:
# Core Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from collections import Counter
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud

# Download NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

## 📊 Data Loading & Initial Exploration

In [ ]:
# Load the dataset
df = pd.read_csv('archive/IMDB Dataset.csv')

print(f"📁 Dataset Shape: {df.shape}")
print(f"📝 Columns: {list(df.columns)}")
print(f"🔍 Data Types:\n{df.dtypes}")
print(f"\n📋 Sample Data:")
df.head()

In [ ]:
# Check for missing values
print("🔍 Missing Values:")
print(df.isnull().sum())

# Check for duplicates
duplicates = df.duplicated().sum()
print(f"\n🔄 Duplicate Rows: {duplicates}")

# Remove duplicates if any
if duplicates > 0:
    df = df.drop_duplicates()
    print(f"✅ Removed {duplicates} duplicate rows")
    print(f"📁 New Shape: {df.shape}")

## 📈 Sentiment Distribution Analysis

In [ ]:
# Sentiment distribution
sentiment_counts = df['sentiment'].value_counts()
print("📊 Sentiment Distribution:")
print(sentiment_counts)

# Create visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Bar plot
sns.countplot(data=df, x='sentiment', ax=ax1)
ax1.set_title('Sentiment Distribution', fontsize=16, fontweight='bold')
ax1.set_xlabel('Sentiment', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)

# Add count labels on bars
for i, count in enumerate(sentiment_counts):
    ax1.text(i, count + 50, str(count), ha='center', fontsize=12, fontweight='bold')

# Pie chart
colors = ['#ff9999', '#66b3ff']
ax2.pie(sentiment_counts, labels=sentiment_counts.index, autopct='%1.1f%%', 
        colors=colors, startangle=90, explode=(0.05, 0.05))
ax2.set_title('Sentiment Percentage', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"🎯 Dataset is {'balanced' if abs(sentiment_counts[0] - sentiment_counts[1]) < 1000 else 'imbalanced'}")

## 📝 Text Analysis - Length & Word Count

In [ ]:
# Calculate text statistics
df['review_length'] = df['review'].str.len()
df['word_count'] = df['review'].str.split().str.len()

print("📊 Text Statistics:")
print(f"Average Review Length: {df['review_length'].mean():.2f} characters")
print(f"Average Word Count: {df['word_count'].mean():.2f} words")
print(f"Min Review Length: {df['review_length'].min()} characters")
print(f"Max Review Length: {df['review_length'].max()} characters")

# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Review length distribution
sns.histplot(data=df, x='review_length', hue='sentiment', kde=True, ax=axes[0,0])
axes[0,0].set_title('Review Length Distribution', fontsize=14, fontweight='bold')
axes[0,0].set_xlabel('Character Count')

# Word count distribution
sns.histplot(data=df, x='word_count', hue='sentiment', kde=True, ax=axes[0,1])
axes[0,1].set_title('Word Count Distribution', fontsize=14, fontweight='bold')
axes[0,1].set_xlabel('Word Count')

# Box plot for review length by sentiment
sns.boxplot(data=df, x='sentiment', y='review_length', ax=axes[1,0])
axes[1,0].set_title('Review Length by Sentiment', fontsize=14, fontweight='bold')

# Box plot for word count by sentiment
sns.boxplot(data=df, x='sentiment', y='word_count', ax=axes[1,1])
axes[1,1].set_title('Word Count by Sentiment', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 🧹 Text Preprocessing Pipeline

In [ ]:
class TextPreprocessor:
    """Comprehensive text preprocessing pipeline for sentiment analysis."""
    
    def __init__(self):
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words('english'))
        # Keep important sentiment words
        self.stop_words.discard('not')
        self.stop_words.discard('no')
        self.stop_words.discard('nor')
        
    def clean_text(self, text):
        """Basic text cleaning."""
        # Remove HTML tags
        text = re.sub(r'<.*?>', '', text)
        # Remove URLs
        text = re.sub(r'http\S+|www\S+', '', text)
        # Remove email addresses
        text = re.sub(r'\S+@\S+', '', text)
        # Remove special characters and numbers (keep letters and spaces)
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        # Convert to lowercase
        text = text.lower()
        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text
    
    def tokenize_and_remove_stopwords(self, text):
        """Tokenize and remove stopwords."""
        tokens = word_tokenize(text)
        # Remove stopwords and short words
        tokens = [token for token in tokens if token not in self.stop_words and len(token) > 2]
        return tokens
    
    def lemmatize_tokens(self, tokens):
        """Lemmatize tokens."""
        return [self.lemmatizer.lemmatize(token) for token in tokens]
    
    def preprocess(self, text):
        """Full preprocessing pipeline."""
        # Step 1: Clean text
        cleaned = self.clean_text(text)
        
        # Step 2: Tokenize and remove stopwords
        tokens = self.tokenize_and_remove_stopwords(cleaned)
        
        # Step 3: Lemmatize
        lemmatized = self.lemmatize_tokens(tokens)
        
        # Join back to string
        return ' '.join(lemmatized)

# Initialize preprocessor
preprocessor = TextPreprocessor()
print("🔧 Text Preprocessor initialized!")

In [ ]:
# Test preprocessing on sample reviews
print("🧪 Testing Preprocessing Pipeline:")
print("=" * 50)

for i in range(3):
    original = df.iloc[i]['review'][:200] + "..."
    sentiment = df.iloc[i]['sentiment']
    
    print(f"\n📝 Sample {i+1} ({sentiment}):")
    print(f"Original: {original}")
    
    processed = preprocessor.preprocess(df.iloc[i]['review'])
    print(f"Processed: {processed[:150]}...")
    print(f"Length reduction: {len(df.iloc[i]['review'])} → {len(processed)} characters")

## 🗂️ Word Frequency Analysis

In [ ]:
# Process all reviews (this might take a moment)
print("🔄 Processing all reviews...")
df['processed_review'] = df['review'].progress_apply(preprocessor.preprocess)

# Separate positive and negative reviews
positive_reviews = df[df['sentiment'] == 'positive']['processed_review']
negative_reviews = df[df['sentiment'] == 'negative']['processed_review']

print("✅ All reviews processed!")

In [ ]:
# Word frequency analysis
def get_word_frequency(reviews, top_n=20):
    """Get top N most frequent words."""
    all_words = ' '.join(reviews).split()
    word_freq = Counter(all_words)
    return word_freq.most_common(top_n)

positive_words = get_word_frequency(positive_reviews)
negative_words = get_word_frequency(negative_reviews)

print("🏆 Top 20 Positive Words:")
for word, count in positive_words:
    print(f"{word}: {count}")

print("\n💔 Top 20 Negative Words:")
for word, count in negative_words:
    print(f"{word}: {count}")

## ☁️ Word Cloud Visualization

In [ ]:
# Generate word clouds
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# Positive word cloud
positive_text = ' '.join(positive_reviews)
wordcloud_pos = WordCloud(width=800, height=400, background_color='white',
                          colormap='Greens', max_words=100).generate(positive_text)

ax1.imshow(wordcloud_pos, interpolation='bilinear')
ax1.set_title('Positive Reviews Word Cloud', fontsize=16, fontweight='bold')
ax1.axis('off')

# Negative word cloud
negative_text = ' '.join(negative_reviews)
wordcloud_neg = WordCloud(width=800, height=400, background_color='white',
                          colormap='Reds', max_words=100).generate(negative_text)

ax2.imshow(wordcloud_neg, interpolation='bilinear')
ax2.set_title('Negative Reviews Word Cloud', fontsize=16, fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.show()

## 💾 Data Preparation for Modeling

In [ ]:
# Encode sentiment labels
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['sentiment_encoded'] = le.fit_transform(df['sentiment'])

print(f"🏷️ Label Encoding:")
print(f"Negative: {le.transform(['negative'])[0]}")
print(f"Positive: {le.transform(['positive'])[0]}")

# Save the processed data
processed_df = df[['processed_review', 'sentiment', 'sentiment_encoded']].copy()
processed_df.to_csv('processed_imdb_data.csv', index=False)

print(f"\n💾 Processed data saved to 'processed_imdb_data.csv'")
print(f"📁 Final dataset shape: {processed_df.shape}")

# Display final sample
print(f"\n📋 Sample of processed data:")
processed_df.head()

## 📊 Summary Statistics

In [ ]:
# Final summary
print("🎯 DATA PREPARATION SUMMARY")
print("=" * 40)
print(f"📁 Dataset Size: {df.shape[0]:,} reviews")
print(f"⚖️  Sentiment Balance: {sentiment_counts['positive']:,} positive, {sentiment_counts['negative']:,} negative")
print(f"📝 Average Review Length: {df['review_length'].mean():.0f} characters")
print(f"🔤 Average Word Count: {df['word_count'].mean():.0f} words")
print(f"🧹 Preprocessing Applied: HTML removal, URL removal, special chars, lowercase, stopwords, lemmatization")
print(f"💾 Output: processed_imdb_data.csv")

print("\n✅ Data preparation complete! Ready for modeling phase.")